# QuantBench requested 模型清单契约 v1

目标：把用户明确点名的 QuantBench 模型清单固化成 `src/model` 的稳定契约，并用默认 matrix 训练、预测、回测和 ledger 证明清单没有遗漏。

边界：本实验不引入强化学习。分类、MSE、IC、ranking 与收益风险 utility 仍由 `src/objective` 统一承接。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

EXPERIMENT_DIR = ROOT / 'research' / 'quantbench_requested_model_contract_v1'
MATRIX_CARD_DIR = EXPERIMENT_DIR / 'matrix_card'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
MATRIX_CARD_DIR.mkdir(parents=True, exist_ok=True)

ROOT

PosixPath('/Users/rziiiii/dev/workspace')

In [2]:
import pandas as pd

from src.model import QUANTBENCH_MODEL_NAMES, REQUESTED_QUANTBENCH_MODEL_NAMES
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.training import model_adapter_name, model_adapter_status

requested = tuple(REQUESTED_QUANTBENCH_MODEL_NAMES)
registered = tuple(QUANTBENCH_MODEL_NAMES)
missing_from_registry = sorted(set(requested) - set(registered))
extra_in_registry = sorted(set(registered) - set(requested))
order_matches = requested == registered

capability_table = model_capability_table(registered)
adapter_audit = pd.DataFrame(
    [
        {
            'model': name,
            'adapter_status': model_adapter_status(name),
            'model_adapter': model_adapter_name(name),
        }
        for name in registered
    ]
).merge(capability_table, on='model', how='left')

contract_summary = {
    'schema_version': 'research.quantbench_requested_model_contract.v1',
    'requested_model_count': len(requested),
    'registered_model_count': len(registered),
    'order_matches': order_matches,
    'missing_from_registry': missing_from_registry,
    'extra_in_registry': extra_in_registry,
    'smoke_adapter_models': sorted(
        adapter_audit.loc[adapter_audit['adapter_status'].eq('smoke_adapter'), 'model'].astype(str).tolist()
    ),
    'fallback_adapter_models': sorted(
        adapter_audit.loc[adapter_audit['model_adapter'].eq('smoke_fallback_v1'), 'model'].astype(str).tolist()
    ),
    'proxy_models': sorted(adapter_audit.loc[adapter_audit['is_proxy'].astype(bool), 'model'].astype(str).tolist()),
    'model_fidelity': model_fidelity_summary(registered),
}

assert not missing_from_registry
assert not extra_in_registry
assert order_matches
assert not contract_summary['smoke_adapter_models']
assert not contract_summary['fallback_adapter_models']
assert not contract_summary['proxy_models']

adapter_audit.to_csv(EXPERIMENT_DIR / 'adapter_audit.csv', index=False)
pd.DataFrame({'model': requested}).to_csv(EXPERIMENT_DIR / 'requested_model_contract.csv', index=False)
contract_summary

{'schema_version': 'research.quantbench_requested_model_contract.v1',
 'requested_model_count': 32,
 'registered_model_count': 32,
 'order_matches': True,
 'missing_from_registry': [],
 'extra_in_registry': [],
 'smoke_adapter_models': [],
 'fallback_adapter_models': [],
 'proxy_models': [],
 'model_fidelity': {'schema_version': 'model.fidelity.v1',
  'model_count': 32,
  'fidelity_counts': {'native_graph': 9,
   'native_library': 10,
   'native_sequence': 7,
   'native_torch': 6},
  'proxy_models': [],
  'native_models': ['alstm_mse',
   'autoformer_mse',
   'catboost_rank',
   'darnn_mse',
   'dlinear_mse',
   'extra_trees_rank',
   'fedformer_mse',
   'gat_mse',
   'gcn_mse',
   'gru_mse',
   'hawkes_gru_mse',
   'hist_gradient_mse',
   'hypergraph_mse',
   'lightgbm_mse',
   'logistic_clf',
   'lstm_mse',
   'market_graph_mse',
   'mlp_mixer_mse',
   'mlp_mse',
   'patch_tst_mse',
   'random_forest_clf',
   'rgcn_mse',
   'ridge_ic',
   'ridge_mse',
   'rsr_mse',
   'sequence_lag_r

In [3]:
from src.model.matrix import run_quantbench_matrix_smoke

outputs = run_quantbench_matrix_smoke(
    card_dir=MATRIX_CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
)

ledger = outputs.model_run_ledger.copy()
ledger_summary = json.loads((MATRIX_CARD_DIR / 'model_run_ledger_summary.json').read_text(encoding='utf-8'))

assert ledger_summary['expected_model_count'] == len(requested)
assert ledger_summary['ledger_model_count'] == len(requested)
assert ledger_summary['registered_model_count'] == len(requested)
assert ledger_summary['trained_model_count'] == len(requested)
assert ledger_summary['backtested_model_count'] == len(requested)
assert ledger_summary['objective_evaluated_model_count'] == len(requested)
assert ledger_summary['promotion_evaluated_model_count'] == len(requested)
assert ledger_summary['analysis_recorded_model_count'] == len(requested)
assert ledger_summary['manifested_model_count'] == len(requested)
assert ledger_summary['missing_models'] == []
assert ledger_summary['all_models_completed'] is True

matrix_summary = {
    'schema_version': 'research.quantbench_requested_model_matrix.v1',
    'matrix_card_dir': str(MATRIX_CARD_DIR.relative_to(ROOT)),
    'model_count': len(requested),
    'ledger_summary': ledger_summary,
    'top_model': str(outputs.model_comparison.iloc[0]['model']),
    'top_return_pct': float(outputs.model_comparison.iloc[0]['return_pct']),
    'top_sharpe': float(outputs.model_comparison.iloc[0]['sharpe']),
    'top_max_drawdown_pct': float(outputs.model_comparison.iloc[0]['max_drawdown_pct']),
    'analysis_layers': outputs.summary['analysis_layers'],
}

summary = {
    **contract_summary,
    'matrix_summary': matrix_summary,
}
(EXPERIMENT_DIR / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

pd.DataFrame([matrix_summary]).drop(columns=['ledger_summary', 'analysis_layers']).to_csv(
    EXPERIMENT_DIR / 'matrix_summary.csv', index=False
)
outputs.model_comparison.head(8)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


,rank,model,return_pct,benchmark_return_pct,relative_return_pct,max_drawdown_pct,sharpe,avg_position,turnover,stability_score
0,1,mlp_mixer_mse,55.285791,58.118381,-2.832590,20.475703,4.121763,0.763458,5.091698,60.725394
1,2,rsr_mse,45.636395,58.118381,-12.481986,22.981182,3.683452,0.747728,5.036027,43.672385
2,3,tcn_mse,44.126973,58.118381,-13.991408,24.842772,3.516625,0.762473,3.515777,41.208206
3,4,mlp_mse,45.180246,58.118381,-12.938135,24.485225,3.757595,0.702538,6.004026,40.141611
4,5,fedformer_mse,19.532160,58.118381,-38.586221,7.586366,3.648972,0.278940,4.978693,36.581539
5,6,darnn_mse,37.592797,58.118381,-20.525584,26.828884,3.161380,0.725121,0.787754,34.094988
6,7,thgnn_mse,44.019103,58.118381,-14.099279,28.171913,3.439516,0.755381,4.754980,33.689408
7,8,hawkes_gru_mse,31.407994,58.118381,-26.710387,14.707499,3.249848,0.643984,6.476610,32.568884


## 中文结论

- 做了什么：新增并验证 `REQUESTED_QUANTBENCH_MODEL_NAMES`，把用户点名的 32 个 QuantBench 模型固定为 `src/model` 的 requested contract；同时运行默认 `run_quantbench_matrix_smoke`，生成 `matrix_card/` 下的训练、预测、回测、objective、promotion、analysis 与 ledger 证据。
- 核心指标：requested model count=32，registered model count=32，ledger trained/backtested/objective/promotion/analysis/manifested model count 均应为 32；`missing_models=[]` 且 `all_models_completed=True`。
- regime 差异：本 notebook 主要验证模型清单和运行完整性；regime/window 细分结果已经写入 `matrix_card/regime_metrics.csv`、`matrix_card/window_metrics.csv`、`matrix_card/window_regime_metrics.csv`、`matrix_card/model_effect_deltas.csv` 和 `matrix_card/model_review_briefing.csv`，供后续 analysis notebook 深挖。
- 是否晋升：不晋升生产策略。本实验是框架契约与证据补强，不改变交易策略。
- 风险：所有深度序列和图模型仍是 compact/native 研究实现，不等同于完整论文级大规模训练；本实验使用 2026-01-01 至 2026-06-09 的短窗口，只能证明框架闭环和相对对比可运行。
- 下一步：把 requested contract 接入 dashboard/schedule health 时，只展示 ledger 完整性和 adapter/fidelity 状态，不把短窗口结果解释为实盘收益结论。